<a href="https://colab.research.google.com/github/Luisen25/ELe/blob/main/ModeloInfusion3poros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
%%html

<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>Modelo de tres poros: infusión y permanencia</title>
<style>
:root{--bg:#fff;--fg:#1d1d1f;--mut:#666;--card:#f3f4f6;--bl:rgba(255,120,120,.16);--fl:rgba(80,160,255,.26);--mem:#555;
box-sizing:border-box;padding-top:env(safe-area-inset-top,0px);padding-bottom:env(safe-area-inset-bottom,0px)}
@media (prefers-color-scheme:dark){:root:not([data-theme="light"]){--bg:#17181b;--fg:#ececec;--mut:#a0a0a0;--card:#24262b;--mem:#9a9a9a;--bl:rgba(255,100,100,.2);--fl:rgba(80,150,255,.3);color-scheme:dark}}
:root[data-theme="dark"]{--bg:#17181b;--fg:#ececec;--mut:#a0a0a0;--card:#24262b;--mem:#9a9a9a;--bl:rgba(255,100,100,.2);--fl:rgba(80,150,255,.3);color-scheme:dark}
html{scroll-padding-top:env(safe-area-inset-top,0px)}
body{margin:0;background:var(--bg);color:var(--fg);font-family:system-ui,-apple-system,"Segoe UI",sans-serif;padding:14px}
h1{font-size:1.15rem;margin:0 0 4px}
p{color:var(--mut);font-size:.85rem;margin:0 0 10px}
canvas{width:100%;max-width:900px;height:auto;display:block;border-radius:10px;background:var(--card)}
.row{display:flex;flex-wrap:wrap;gap:14px;margin:10px 0;max-width:900px;font-size:.85rem;align-items:center}
.row label{display:flex;flex-direction:column;gap:2px;min-width:190px}
.leg{display:flex;flex-wrap:wrap;gap:12px;max-width:900px;font-size:.82rem}
.leg span{display:flex;align-items:center;gap:5px}
.dot{width:11px;height:11px;border-radius:50%;display:inline-block}
#st{font-size:.9rem;font-weight:600;margin:6px 0;max-width:900px}
button{padding:6px 12px;border-radius:8px;border:1px solid var(--mut);background:var(--card);color:var(--fg);cursor:pointer}
</style>
</head>
<body>
<h1>Modelo de tres poros: infusión y permanencia</h1>
<p>Primero se infunde el dializado por el catéter y llena la cavidad peritoneal; después, el agua y los solutos cruzan por los poros.</p>
<canvas id="c" width="900" height="500"></canvas>
<div id="st"></div>
<div class="row">
  <label>Velocidad de infusión<input id="inf" type="range" min="2" max="15" value="5"></label>
  <label>Glucosa del dializado (empuje osmótico)<input id="gl" type="range" min="0" max="100" value="40"></label>
  <label>Agitación térmica<input id="sp" type="range" min="30" max="200" value="100"></label>
  <button id="pl">Pausa</button><button id="rs">Reiniciar infusión</button>
</div>
<div class="leg" id="leg"></div>
<script>
const cv=document.getElementById('c'),ctx=cv.getContext('2d'),W=900,H=500,MX=450,MW=8,TOP=60,CX=860;
const pores=[{y:140,h:9,n:'Ultrapequeño (acuaporina)'},{y:270,h:24,n:'Pequeño'},{y:420,h:42,n:'Grande'}];
const T=[
 {k:'Agua',n:160,c:'#4aa3ff',r:3,ok:[0,1,2],st:1.0},
 {k:'Urea',n:45,c:'#2ca02c',r:5,ok:[1,2],st:0.8},
 {k:'Glucosa',n:45,c:'#ff9f1c',r:6,ok:[1,2],st:0.7},
 {k:'Albúmina',n:10,c:'#d62728',r:11,ok:[2],st:0.45}];
let P=[],Q=[],spawned=0,level=0,run=true;
const rnd=(a,b)=>a+Math.random()*(b-a);
const gauss=()=>(Math.random()+Math.random()+Math.random()-1.5)*1.6;
const surf=()=>H-level*(H-TOP);
function init(){
 P=[];level=0;spawned=0;Q=[];
 const left=[[0,80],[1,45],[3,10]];
 left.forEach(([ti,n])=>{for(let i=0;i<n;i++)P.push({t:ti,x:rnd(15,MX-30),y:rnd(25,H-15),vx:0,vy:0,drop:false})});
 for(let i=0;i<80;i++)Q.push(0);for(let i=0;i<45;i++)Q.push(2);
 Q.sort(()=>Math.random()-.5)}
function spawn(ti){P.push({t:ti,x:CX+rnd(-4,4),y:TOP-8,vx:0,vy:2,drop:true})}
function passable(y,r,ok,s){return y>=s&&ok.some(i=>Math.abs(y-pores[i].y)<=pores[i].h-r*0.5)}
function step(){
 const sp=document.getElementById('sp').value/100,dr=level>.05?document.getElementById('gl').value/100*0.9:0,s=surf();
 for(const p of P){const t=T[p.t];
  if(p.drop){p.y+=2.4;p.x=Math.min(W-t.r,Math.max(MX+MW+t.r,p.x+gauss()*.6));if(p.y>=s)p.drop=false;continue}
  p.vx=p.vx*.8+gauss()*t.st*sp*1.6;p.vy=p.vy*.8+gauss()*t.st*sp*1.6;
  let nx=p.x+p.vx+(t.k==='Agua'?dr:0),ny=p.y+p.vy;
  if(nx<t.r){nx=t.r;p.vx=Math.abs(p.vx)}if(nx>W-t.r){nx=W-t.r;p.vx=-Math.abs(p.vx)}
  if(ny<t.r){ny=t.r;p.vy=Math.abs(p.vy)}if(ny>H-t.r){ny=H-t.r;p.vy=-Math.abs(p.vy)}
  const inOld=Math.abs(p.x-MX)<MW+t.r*.3,inNew=Math.abs(nx-MX)<MW+t.r*.3;
  if(inNew&&!passable(ny,t.r,t.ok,s)){if(inOld){ny=p.y;p.vy=-p.vy}else{nx=p.x;p.vx=-p.vx}}
  if(nx>MX+MW&&ny<s+t.r){ny=Math.min(H-t.r,s+t.r);p.vy=Math.abs(p.vy)}
  p.x=nx;p.y=ny}}
function draw(){
 const cs=getComputedStyle(document.documentElement),s=surf(),fg=cs.getPropertyValue('--fg'),mem=cs.getPropertyValue('--mem');
 ctx.clearRect(0,0,W,H);
 ctx.fillStyle=cs.getPropertyValue('--bl');ctx.fillRect(0,0,MX-MW,H);
 ctx.fillStyle=cs.getPropertyValue('--fl');ctx.fillRect(MX+MW,s,W-MX-MW,H-s);
 if(level>0){ctx.strokeStyle='#4aa3ff';ctx.lineWidth=2;ctx.beginPath();ctx.moveTo(MX+MW,s);ctx.lineTo(W,s);ctx.stroke()}
 ctx.fillStyle=mem;let prev=0;
 [...pores].sort((a,b)=>a.y-b.y).forEach(o=>{ctx.fillRect(MX-MW,prev,MW*2,o.y-o.h-prev);prev=o.y+o.h});
 ctx.fillRect(MX-MW,prev,MW*2,H-prev);
 ctx.fillRect(CX-5,0,10,TOP-4);
 if(level<1){ctx.fillStyle='#4aa3ff';ctx.fillRect(CX-2,0,4,TOP-4)}
 ctx.fillStyle=fg;ctx.font='12px system-ui';ctx.textAlign='center';
 ctx.fillText('CAPILAR (sangre)',MX/2,16);ctx.fillText('CAVIDAD PERITONEAL',MX+MX/2-40,16);
 ctx.textAlign='right';ctx.fillText('Catéter: entra dializado',CX-12,34);ctx.textAlign='center';
 pores.forEach(o=>ctx.fillText(o.n,MX,o.y-o.h-6));
 for(const p of P){const t=T[p.t];ctx.fillStyle=t.c;ctx.beginPath();ctx.arc(p.x,p.y,t.r,0,6.283);ctx.fill()}
 document.getElementById('st').textContent=level<1?`Fase 1: infusión del dializado (${Math.round(level*100)} % del volumen)`:'Fase 2: permanencia (los poros transportan agua y solutos)';
 document.getElementById('leg').innerHTML=T.map((t,i)=>{const d=P.filter(p=>p.t===i&&p.x>MX&&!p.drop).length;
  return `<span><i class="dot" style="background:${t.c}"></i>${t.k}: ${d}/${t.n} en la cavidad</span>`}).join('')}
function loop(){
 if(run){if(level<1){level=Math.min(1,level+document.getElementById('inf').value*0.0003);
  while(spawned<Math.floor(level*Q.length))spawn(Q[spawned++])}step()}
 draw();requestAnimationFrame(loop)}
document.getElementById('pl').onclick=e=>{run=!run;e.target.textContent=run?'Pausa':'Continuar'};
document.getElementById('rs').onclick=init;
init();loop();
</script>
</body>
</html>





